# Cuaderno 4 · Dataset etiquetado con incendios reales (v2)

En la v1 (Cuadernos 1–3) la variable objetivo era un índice calculado con una fórmula a partir de las mismas variables que recibía la red. El modelo aprendía a imitar la fórmula, no a anticipar incendios, y por eso el tribunal concluyó que el resultado era un *prototipo de susceptibilidad, no una alerta*.

En la v2 la variable objetivo procede de **incendios observados**:

| | v1 (TFM) | v2 |
|---|---|---|
| Unidad de análisis | Estación de AEMET en una fecha | Punto con vegetación en una fecha |
| Etiqueta | Índice 0,35·meteo + 0,65·NDMI | **1 si ese día empezó un incendio ahí, 0 si no** |
| Origen de la etiqueta | Fórmula propia | Focos térmicos FIRMS (NASA), agrupados en eventos |
| Imagen Sentinel-2 | Del mismo día, sin máscara de nubes | Mediana **sin nubes** (SCL) de los 30 días **previos** |
| Meteorología | Estaciones AEMET (58 puntos) | ERA5-Land diario (cualquier punto, mismo producto en todo el periodo) |
| Ubicaciones | Solo alrededor de las estaciones | Todo el territorio con vegetación |

**Pasos:**
1. Descargar los focos FIRMS de 2017 a 2026 sobre Canarias.
2. Quitar falsos incendios (erupción de La Palma 2021) y agrupar los focos en **eventos**.
3. Generar **negativos**: puntos con vegetación y fechas sin incendios cerca en espacio y tiempo.
4. Calcular en Earth Engine las variables de cada punto (Sentinel-2, ERA5-Land, relieve, cobertura).
5. Unir todo en `dataset_v2.csv` para el `Cuaderno5`.

## 1. Entorno y parámetros

In [ ]:
import os, sys, time, glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

if not os.path.exists("/content/repo"):
    !git clone -q https://github.com/joseandres03/sentinel2-meteo_IA_wildfire_risk.git /content/repo
sys.path.insert(0, "/content/repo/src")

from google.colab import drive
drive.mount("/content/drive")

from wildfire import gee, etiquetas, validacion, meteo

PROYECTO_GCP = "tfm-bbdd-499813"
CARPETA_DRIVE = "TFM_Riesgo_Incendios_Canarias_v2"          # carpeta de exportación de Earth Engine
RUTA_TRABAJO = "/content/drive/MyDrive/TFM_Riesgo_Incendios_Canarias/v2"
os.makedirs(RUTA_TRABAJO, exist_ok=True)

FECHA_INICIO = "2017-04-01"     # Sentinel-2 L2A armonizado disponible de forma continua
FECHA_FIN = "2026-06-30"        # ERA5-Land tiene unos días de retraso
CONFIANZA_MIN_FIRMS = 50        # % de confianza mínimo del foco
RADIO_EVENTO_KM = 2.0           # focos a menos de 2 km ...
VENTANA_EVENTO_DIAS = 3         # ... y 3 días forman el mismo incendio
NEGATIVOS_POR_INCENDIO = 20
RADIO_ENTORNO_M = 500          # radio del entorno de cada punto (≈ incertidumbre de un foco MODIS de 1 km)
PREFIJO = "caract_r500"        # nombre de las exportaciones (cambiarlo evita mezclar con ejecuciones anteriores)
EXCLUSION_KM = 5.0              # un negativo no puede estar a menos de 5 km ...
EXCLUSION_DIAS = 30             # ... y 30 días de un incendio
SEMILLA = 42

gee.inicializar(PROYECTO_GCP)

## 2. Focos térmicos FIRMS

FIRMS (NASA) registra a diario los píxeles de 1 km en los que los satélites MODIS detectan una anomalía térmica compatible con fuego. Descargo los centroides de los focos año a año para no superar los límites de Earth Engine.

In [ ]:
columnas = ["lat", "lon", "fecha"]
anuales = []
for anio in range(int(FECHA_INICIO[:4]), int(FECHA_FIN[:4]) + 1):
    ini = max(f"{anio}-01-01", FECHA_INICIO)
    fin = min(f"{anio + 1}-01-01", FECHA_FIN)
    fc = gee.focos_firms(ini, fin, confianza_min=CONFIANZA_MIN_FIRMS)
    df_anio = gee.fc_a_dataframe(fc, columnas)
    print(f"{anio}: {len(df_anio)} focos")
    anuales.append(df_anio)

focos = pd.concat(anuales, ignore_index=True)
focos["fuente"] = "MODIS"
focos.to_csv(os.path.join(RUTA_TRABAJO, "focos_firms_modis.csv"), index=False)
print("Total de focos MODIS:", len(focos))

### Añadir focos VIIRS de 375 m (recomendado)

MODIS mira píxeles de 1 km y solo detecta los incendios más grandes. **VIIRS** mira píxeles de 375 m y detecta muchos más incendios pequeños, así que es la mejor forma de ampliar el dataset. Su archivo histórico se pide gratis en [FIRMS · Archive Download](https://firms.modaps.eosdis.nasa.gov/download/) y llega por correo como un ZIP con uno o varios CSV (`fire_archive_...` y, para los meses más recientes, `fire_nrt_...`).

Sube **todos los CSV del ZIP** a la carpeta `v2/viirs` de tu Drive. Esta celda los lee y los une.

In [ ]:
RUTA_VIIRS = os.path.join(RUTA_TRABAJO, "viirs")
archivos_viirs = sorted(glob.glob(os.path.join(RUTA_VIIRS, "*.csv")))
if archivos_viirs:
    viirs = pd.concat([pd.read_csv(a) for a in archivos_viirs], ignore_index=True)
    viirs = viirs.rename(columns={"latitude": "lat", "longitude": "lon", "acq_date": "fecha"})
    # confianza VIIRS: 'l' (baja), 'n' (nominal), 'h' (alta) -> descarto las de baja confianza
    if "confidence" in viirs:
        viirs = viirs[viirs["confidence"].astype(str).str.lower().str[0] != "l"]
    # 'type' (solo en el archivo histórico): 0 = vegetación, 1 = volcán, 2 = fuente estática (industria), 3 = mar
    if "type" in viirs:
        viirs = viirs[viirs["type"].fillna(0).astype(int) == 0]
    viirs = viirs[(viirs["fecha"] >= FECHA_INICIO) & (viirs["fecha"] <= FECHA_FIN)]
    en_canarias = validacion.isla_de(viirs["lat"], viirs["lon"]) != "Fuera"
    viirs = viirs.loc[en_canarias, ["lat", "lon", "fecha"]].drop_duplicates().assign(fuente="VIIRS")
    print(f"{len(archivos_viirs)} archivos VIIRS · {len(viirs)} focos en Canarias")
    focos = pd.concat([focos, viirs], ignore_index=True)
    print(f"Total de focos (MODIS + VIIRS): {len(focos)}")
else:
    print(f"Sin archivos en {RUTA_VIIRS}: se continúa solo con MODIS.")

## 3. De focos a eventos de incendio

1. Elimino los focos de la **erupción de Cumbre Vieja** (La Palma, 2021), que el satélite detecta como fuego.
2. Agrupo los focos a menos de 2 km y 3 días en un mismo **evento**. La fecha del evento es la de su primera detección, que es lo que queremos anticipar.

In [ ]:
focos["fecha"] = pd.to_datetime(focos["fecha"])
antes = len(focos)
focos = etiquetas.filtrar_exclusiones(focos)
print(f"Focos descartados por exclusiones: {antes - len(focos)}")

eventos = etiquetas.agrupar_eventos(focos, radio_km=RADIO_EVENTO_KM, ventana_dias=VENTANA_EVENTO_DIAS)
eventos["isla"] = validacion.isla_de(eventos["lat"], eventos["lon"])
eventos = eventos[eventos["isla"] != "Fuera"].reset_index(drop=True)
eventos.to_csv(os.path.join(RUTA_TRABAJO, "eventos_incendio.csv"), index=False)
print(f"{len(eventos)} eventos de incendio")

fig, ejes = plt.subplots(1, 3, figsize=(16, 4))
eventos["fecha"].dt.year.value_counts().sort_index().plot.bar(ax=ejes[0], title="Eventos por año")
eventos["fecha"].dt.month.value_counts().sort_index().plot.bar(ax=ejes[1], title="Eventos por mes")
eventos["isla"].value_counts().plot.barh(ax=ejes[2], title="Eventos por isla")
plt.tight_layout(); plt.show()

## 4. Negativos: puntos y fechas sin incendio

Tomo una muestra de puntos sobre **tierra firme, con cualquier tipo de cobertura**, y les asigno fechas aleatorias del periodo. Descarto los que caen a menos de 5 km y 30 días de un incendio.

¿Por qué en cualquier cobertura y no solo sobre vegetación? Los focos de incendio pueden caer en cualquier sitio, también cerca de pueblos o cultivos. Si los "no incendios" se tomaran solo sobre vegetación, el modelo aprendería a distinguir "zona con o sin vegetación" en lugar de riesgo de incendio. Así, las dos clases se eligen con el mismo criterio y luego pasan **el mismo filtro de vegetación** (apartado 6).

Las fechas se reparten por todo el año para que el modelo pueda aprender la estacionalidad real del riesgo.

In [ ]:
n_negativos = NEGATIVOS_POR_INCENDIO * len(eventos)
pool = gee.fc_a_dataframe(gee.puntos_tierra(n=4900, semilla=SEMILLA), ["lat", "lon"])
print(f"Puntos de tierra disponibles: {len(pool)}")

negativos = etiquetas.muestrear_negativos(
    pool, eventos, n=n_negativos, fecha_inicio=FECHA_INICIO, fecha_fin=FECHA_FIN,
    exclusion_km=EXCLUSION_KM, exclusion_dias=EXCLUSION_DIAS, semilla=SEMILLA)

dataset = etiquetas.construir_dataset(eventos, negativos)
dataset.insert(0, "id", np.arange(len(dataset)))
dataset["isla"] = validacion.isla_de(dataset["lat"], dataset["lon"])
dataset["bloque_10km"] = validacion.bloques_espaciales(dataset["lat"], dataset["lon"], 10)
dataset.to_csv(os.path.join(RUTA_TRABAJO, "etiquetas_v2.csv"), index=False)
dataset["incendio"].value_counts()

## 5. Variables de cada punto (cálculo en Earth Engine)

Para cada punto y fecha se calculan en los servidores de Earth Engine:

- **Sentinel-2:** mediana sin nubes (SCL) de los 30 días **anteriores**: bandas, NDVI, NDMI, NBR y NDWI en un radio de 320 m.
- **ERA5-Land:** temperatura máxima, temperatura y punto de rocío medios, viento (u, v) del día y lluvia acumulada en los 30 días previos.
- **Relieve (SRTM):** altitud, pendiente y orientación.
- **Cobertura (ESA WorldCover):** fracción de arbolado, matorral, pastizal, cultivo y zona urbana.

El cálculo se lanza como tareas de exportación a Google Drive. Se puede seguir en el [gestor de tareas de Earth Engine](https://code.earthengine.google.com/tasks). Según el tamaño, puede tardar desde minutos hasta algunas horas.

In [ ]:
tareas = gee.exportar_caracteristicas(dataset[["id", "lat", "lon", "fecha", "incendio"]],
                                      carpeta_drive=CARPETA_DRIVE, prefijo=PREFIJO,
                                      radio_m=RADIO_ENTORNO_M)
print(f"{len(tareas)} tareas lanzadas")

### Esperar a que terminen las tareas

Esta celda consulta a Earth Engine cada minuto y no termina hasta que todas las tareas acaban. Funciona aunque se haya reiniciado la sesión de Colab, porque busca las tareas por su nombre. Si alguna falla, muestra el motivo.

In [ ]:
import ee

PREFIJO_TAREAS = PREFIJO
TERMINADOS = {"COMPLETED", "FAILED", "CANCELLED"}

def estado_tareas(prefijo=PREFIJO_TAREAS):
    # Última tarea de cada nombre (por si se relanzó alguna)
    ultimas = {}
    for t in ee.batch.Task.list():
        s = t.status()
        nombre = s.get("description", "")
        if nombre.startswith(prefijo) and nombre not in ultimas:
            ultimas[nombre] = s
    return ultimas

inicio = time.time()
while True:
    estados = estado_tareas()
    conteo = pd.Series([s["state"] for s in estados.values()]).value_counts().to_dict()
    print(f"[{(time.time() - inicio) / 60:5.1f} min] {conteo}")
    if estados and all(s["state"] in TERMINADOS for s in estados.values()):
        break
    time.sleep(60)

fallidas = {n: s.get("error_message") for n, s in estados.items() if s["state"] != "COMPLETED"}
if fallidas:
    for nombre, error in fallidas.items():
        print(f"❌ {nombre}: {error}")
    raise RuntimeError("Hay tareas que no se completaron: revisa los errores antes de seguir.")
print(f"✅ {len(estados)} tareas completadas")

## 6. Unión y limpieza del dataset final

In [ ]:
# Si la sesión se reinició, recupero las etiquetas guardadas en el apartado 4
if "dataset" not in globals():
    dataset = pd.read_csv(os.path.join(RUTA_TRABAJO, "etiquetas_v2.csv"), parse_dates=["fecha"])
    print(f"Etiquetas recuperadas de Drive: {len(dataset)} filas")

# Colab no siempre ve los archivos que Earth Engine escribe en Drive después de montarlo:
# vuelvo a montar Drive para refrescar la vista.
drive.mount("/content/drive", force_remount=True)

# Earth Engine crea la carpeta CARPETA_DRIVE en la raíz de "Mi unidad"; si ya existía una
# carpeta con ese nombre puede crear otra igual, así que busco en todo Drive.
archivos = sorted(glob.glob(f"/content/drive/MyDrive/**/{PREFIJO}_*.csv", recursive=True))
print(f"{len(archivos)} archivos exportados")
for a in archivos:
    print("  ", a)
if not archivos:
    raise FileNotFoundError(
        "No hay CSV de características en Drive. Comprueba que la celda anterior terminó con "
        "'tareas completadas' y busca en Google Drive (web) archivos que empiecen por '" + PREFIJO + "'. "
        "Si los ves en la web pero no aquí, espera un par de minutos y vuelve a ejecutar esta celda.")

carac = pd.concat([pd.read_csv(a) for a in archivos], ignore_index=True)
carac = carac.drop(columns=[c for c in ("system:index", ".geo", "lat", "lon", "fecha", "incendio") if c in carac])
final = dataset.merge(carac, on="id", how="left")
final = meteo.derivar_meteo_era5(final)

# Calidad: imágenes Sentinel-2 disponibles y vegetación real en el entorno
final["frac_vegetacion_wc"] = final[["frac_arbolado", "frac_matorral", "frac_pastizal"]].sum(axis=1)

# Comprobación del sesgo: las dos clases deberían tener coberturas comparables antes de filtrar
print("Cobertura media del entorno por clase (0 = sin incendio, 1 = incendio):")
print(final.groupby("incendio")[["frac_vegetacion_wc", "frac_urbano", "frac_cultivo"]].mean().round(3))
sin_s2 = final["ndmi"].isna()
sin_veg = final["frac_vegetacion_wc"] < 0.1
print(f"Sin imagen Sentinel-2 despejada: {sin_s2.sum()} ({sin_s2.mean():.1%})")
print(f"Con menos de un 10 % de vegetación: {sin_veg.sum()} "
      f"(incendios: {(sin_veg & (final['incendio'] == 1)).sum()}; posibles quemas agrícolas o urbanas)")

final = final[~sin_veg].reset_index(drop=True)
final.to_csv(os.path.join(RUTA_TRABAJO, "dataset_v2.csv"), index=False)
print(f"dataset_v2.csv: {len(final)} filas · {final['incendio'].sum()} incendios")
final.describe().T.round(2)

## 7. (Opcional) Parches de 64×64 para la red convolucional

Para comparar los modelos tabulares con una CNN *late fusion* como la de la v1, descargo los parches de Sentinel-2 previos a cada fecha. Es lento (una petición por punto), así que se guarda cada parche en Drive y se puede reanudar.

In [ ]:
RUTA_PARCHES = os.path.join(RUTA_TRABAJO, "parches")
os.makedirs(RUTA_PARCHES, exist_ok=True)

pendientes = [r for r in final.itertuples() if not os.path.exists(os.path.join(RUTA_PARCHES, f"{r.id}.npy"))]
print(f"Parches pendientes: {len(pendientes)}")
fallos = 0
for i, r in enumerate(pendientes):
    parche = gee.extraer_parche(r.lat, r.lon, r.fecha.strftime("%Y-%m-%d"))
    if parche is None:
        fallos += 1
        continue
    np.save(os.path.join(RUTA_PARCHES, f"{r.id}.npy"), parche)
    if i % 200 == 0:
        print(f"{i}/{len(pendientes)} · fallos {fallos}")
print("Fallos:", fallos)